# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a dataset defined by a Croissant schema using the `mlcroissant` library. The dataset investigates protein abundance, modifications, and sequences in human samples via mass spectrometry analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

# Print dataset name and description
print(f"{getattr(metadata_obj, 'name', 'Unknown name')}: {getattr(metadata_obj, 'description', 'No description')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

*Each entity (record set, field, column, etc.) is referenced by its `@id` as per Croissant best practices.*

In [ ]:
# List all record sets and their fields by @id for precise referencing
record_set_ids = [rs['@id'] for rs in getattr(metadata_obj, 'record_sets', [])]
if not record_set_ids:
    # Sometimes Croissant schema uses .records attribute
    try:
        record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
    except Exception:
        record_set_ids = []

if not record_set_ids:
    # Try fallback: scan for common names like 'recordSets', 'recordSet', etc.
    meta_dict = dataset.metadata.to_json()
    for k in ['recordSets', 'recordSet', 'record_sets']:
        if k in meta_dict and isinstance(meta_dict[k], list):
            record_set_ids = [rs['@id'] for rs in meta_dict[k] if '@id' in rs]
            break

if not record_set_ids:
    print("No record sets found in metadata.")
else:
    print("Available record sets by @id:")
    for rs_id in record_set_ids:
        print(f"  - {rs_id}")
    print()
    # Show field ids for each record set
    meta_dict = dataset.metadata.to_json()
    print("Fields (columns/attributes) for each record set, referenced by @id:")
    for rs in meta_dict.get('recordSet', []):
        if '@id' in rs:
            print(f"Record set: {rs['@id']}")
            if 'field' in rs and isinstance(rs['field'], list):
                for field in rs['field']:
                    if isinstance(field, dict) and '@id' in field:
                        print(f"    Field: {field['@id']}")
                    elif isinstance(field, str):
                        print(f"    Field: {field}")
            print()
    # Save out first record set id for use in later steps
    first_record_set_id = record_set_ids[0]

To sample records, below is a preview for the first found record set (referenced by its `@id`):

In [ ]:
# Print sample records for the first record set (by @id)
if record_set_ids:
    for i, record in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# Extract all data from each record set into a DataFrame, keys by record set @id
if not record_set_ids:
    print("No record sets to process.")
else:
    dataframes = {}
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Record set {rs_id} columns: ", df.columns.tolist())
            display(df.head(3))
        else:
            print(f"Record set {rs_id} has no records.")
    # Select the first populated DataFrame for analysis
    main_record_set_id = next(iter(dataframes.keys()))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data, or grouping by key attributes.

In [ ]:
import numpy as np
# Infer candidate numeric fields from the DataFrame columns
df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_fields:
    # Try to coerce floats if all are object type
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

if not numeric_fields:
    print('No numeric fields found for EDA.')
else:
    print(f"Numeric fields: {numeric_fields}")
    numeric_field = numeric_fields[0]  # Pick the first numeric field
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (mean):")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try grouping
    # Pick any non-numeric field as categorical
    group_field = None
    for c in df.columns:
        if c not in numeric_fields:
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show a histogram of the selected numeric field for the filtered records, and a boxplot by the group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print('No numeric field available for plotting.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], kde=True, ax=ax)
    ax.set_title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    # Boxplot by group_field (if available and not too many unique values)
    if group_field and filtered_df[group_field].nunique() < 25:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to discover, extract, and analyze data from the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their Croissant `@id`, supporting reproducible FAIR data workflows. Continue your exploration by referencing other fields or performing advanced statistical modeling.